## Machine Learning Strategies (1) - Model Fitting/Modellanpassung

In [41]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier # added (from sklearn v. 1.7)
import tpqoa
from datetime import datetime, timezone, timedelta
import time
import pickle
import warnings
warnings.filterwarnings('ignore')

In [42]:
api = tpqoa.tpqoa(r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\oanda.cfg')

In [3]:
data=api.get_history(instrument = 'EUR_USD', start = '2025-06-01', end = '2025-12-17', granularity = 'S5', price = 'M')


In [4]:
data

,o,h,l,c,volume,complete
time,,,,,,
2025-06-01 21:04:55,1.13491,1.13491,1.13491,1.13491,1,True
2025-06-01 21:05:00,1.13487,1.13491,1.13487,1.13491,2,True
2025-06-01 21:05:10,1.13487,1.13487,1.13487,1.13487,1,True
2025-06-01 21:05:25,1.13483,1.13483,1.13483,1.13483,1,True
2025-06-01 21:05:40,1.13480,1.13480,1.13478,1.13478,3,True
...,...,...,...,...,...,...
2025-12-16 23:59:10,1.17496,1.17496,1.17494,1.17494,2,True
2025-12-16 23:59:15,1.17494,1.17494,1.17494,1.17494,2,True
2025-12-16 23:59:20,1.17495,1.17496,1.17495,1.17495,5,True


In [5]:
data.drop(['o','h','l','volume','complete'], axis=1, inplace=True)

In [6]:
data

,c
time,
2025-06-01 21:04:55,1.13491
2025-06-01 21:05:00,1.13491
2025-06-01 21:05:10,1.13487
2025-06-01 21:05:25,1.13483
2025-06-01 21:05:40,1.13478
...,...
2025-12-16 23:59:10,1.17494
2025-12-16 23:59:15,1.17494
2025-12-16 23:59:20,1.17495


In [7]:
data.rename(columns={"c": "price"}, inplace=True)

In [52]:
data

,price
time,
2021-01-10 22:00:00,1.22151
2021-01-10 22:05:00,1.22165
2021-01-10 22:10:00,1.22266
2021-01-10 22:15:00,1.22186
2021-01-10 22:20:00,1.22188
...,...
2025-12-16 23:35:00,1.17500
2025-12-16 23:40:00,1.17512
2025-12-16 23:45:00,1.17496


In [6]:
#data.to_csv(r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\20251217_five_minute.csv')

data.to_csv(r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\20251217_five_seconds.csv')

NameError: name 'data' is not defined

# Daten laden

In [43]:
#data = pd.read_csv(r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\20251217_five_minute.csv', parse_dates = ["time"], index_col = "time")

data = pd.read_csv(r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\20251217_five_seconds.csv', parse_dates = ["time"], index_col = "time")

In [44]:
data["returns"] = np.log(data.div(data.shift(1)))

In [45]:
data

,price,returns
time,,
2025-06-01 21:04:55,1.13491,NaN
2025-06-01 21:05:00,1.13491,0.000000
2025-06-01 21:05:10,1.13487,-0.000035
2025-06-01 21:05:25,1.13483,-0.000035
2025-06-01 21:05:40,1.13478,-0.000044
...,...,...
2025-12-16 23:59:10,1.17494,-0.000009
2025-12-16 23:59:15,1.17494,0.000000
2025-12-16 23:59:20,1.17495,0.000009


In [46]:
data.dropna(inplace = True)

In [47]:
data

,price,returns
time,,
2025-06-01 21:05:00,1.13491,0.000000
2025-06-01 21:05:10,1.13487,-0.000035
2025-06-01 21:05:25,1.13483,-0.000035
2025-06-01 21:05:40,1.13478,-0.000044
2025-06-01 21:05:45,1.13478,0.000000
...,...,...
2025-12-16 23:59:10,1.17494,-0.000009
2025-12-16 23:59:15,1.17494,0.000000
2025-12-16 23:59:20,1.17495,0.000009


In [48]:
data["direction"] = np.sign(data.returns)

In [49]:
data

,price,returns,direction
time,,,
2025-06-01 21:05:00,1.13491,0.000000,0.0
2025-06-01 21:05:10,1.13487,-0.000035,-1.0
2025-06-01 21:05:25,1.13483,-0.000035,-1.0
2025-06-01 21:05:40,1.13478,-0.000044,-1.0
2025-06-01 21:05:45,1.13478,0.000000,0.0
...,...,...,...
2025-12-16 23:59:10,1.17494,-0.000009,-1.0
2025-12-16 23:59:15,1.17494,0.000000,0.0
2025-12-16 23:59:20,1.17495,0.000009,1.0


In [50]:
#lags = 2
lags= 5

In [51]:
cols = []
for lag in range(1, lags + 1):
    col = f'lag{lag}'
    data[col] = data.returns.shift(lag)
    cols.append(col)
data.dropna(inplace = True)

In [52]:
means = data[cols].mean()
means

# Ausgabe lag1: -0,0000003117767
# Ausgabe lag2: -0,0000003115345

lag1    1.664405e-08
lag2    1.662704e-08
lag3    1.660610e-08
lag4    1.658923e-08
lag5    1.659330e-08
dtype: float64

In [53]:
data

,price,returns,direction,lag1,lag2,lag3,lag4,lag5
time,,,,,,,,
2025-06-01 21:05:55,1.13478,0.000000,0.0,0.000000,-0.000044,-0.000035,-0.000035,0.000000
2025-06-01 21:06:20,1.13479,0.000009,1.0,0.000000,0.000000,-0.000044,-0.000035,-0.000035
2025-06-01 21:06:45,1.13479,0.000000,0.0,0.000009,0.000000,0.000000,-0.000044,-0.000035
2025-06-01 21:06:50,1.13477,-0.000018,-1.0,0.000000,0.000009,0.000000,0.000000,-0.000044
2025-06-01 21:07:00,1.13476,-0.000009,-1.0,-0.000018,0.000000,0.000009,0.000000,0.000000
...,...,...,...,...,...,...,...,...
2025-12-16 23:59:10,1.17494,-0.000009,-1.0,-0.000009,0.000000,0.000000,0.000017,0.000017
2025-12-16 23:59:15,1.17494,0.000000,0.0,-0.000009,-0.000009,0.000000,0.000000,0.000017
2025-12-16 23:59:20,1.17495,0.000009,1.0,0.000000,-0.000009,-0.000009,0.000000,0.000000


In [54]:
data[cols]

,lag1,lag2,lag3,lag4,lag5
time,,,,,
2025-06-01 21:05:55,0.000000,-0.000044,-0.000035,-0.000035,0.000000
2025-06-01 21:06:20,0.000000,0.000000,-0.000044,-0.000035,-0.000035
2025-06-01 21:06:45,0.000009,0.000000,0.000000,-0.000044,-0.000035
2025-06-01 21:06:50,0.000000,0.000009,0.000000,0.000000,-0.000044
2025-06-01 21:07:00,-0.000018,0.000000,0.000009,0.000000,0.000000
...,...,...,...,...,...
2025-12-16 23:59:10,-0.000009,0.000000,0.000000,0.000017,0.000017
2025-12-16 23:59:15,-0.000009,-0.000009,0.000000,0.000000,0.000017
2025-12-16 23:59:20,0.000000,-0.000009,-0.000009,0.000000,0.000000


In [55]:
stand_devs = data[cols].std()
stand_devs

lag1    0.000035
lag2    0.000035
lag3    0.000035
lag4    0.000035
lag5    0.000035
dtype: float64

In [56]:
data[cols] = (data[cols]-means) / stand_devs
data

,price,returns,direction,lag1,lag2,lag3,lag4,lag5
time,,,,,,,,
2025-06-01 21:05:55,1.13478,0.000000,0.0,-0.000472,-1.249385,-0.999562,-0.999526,-0.000470
2025-06-01 21:06:20,1.13479,0.000009,1.0,-0.000472,-0.000471,-1.249384,-0.999561,-0.999526
2025-06-01 21:06:45,1.13479,0.000000,0.0,0.249316,-0.000471,-0.000471,-1.249384,-0.999561
2025-06-01 21:06:50,1.13477,-0.000018,-1.0,-0.000472,0.249316,-0.000471,-0.000470,-1.249384
2025-06-01 21:07:00,1.13476,-0.000009,-1.0,-0.500049,-0.000471,0.249316,-0.000470,-0.000470
...,...,...,...,...,...,...,...,...
2025-12-16 23:59:10,1.17494,-0.000009,-1.0,-0.241719,-0.000471,-0.000471,0.482026,0.482034
2025-12-16 23:59:15,1.17494,0.000000,0.0,-0.241721,-0.241719,-0.000471,-0.000470,0.482026
2025-12-16 23:59:20,1.17495,0.000009,1.0,-0.000472,-0.241721,-0.241718,-0.000470,-0.000470


In [57]:
# lm = LogisticRegression(C = 1e6, max_iter = 100000, multi_class = "ovr") # old
lm = OneVsRestClassifier(LogisticRegression(C=1e6, max_iter=100000))  # new (from sklearn v. 1.7)

In [16]:
lm

,estimator,LogisticRegre...x_iter=100000)
,n_jobs,None
,verbose,0
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1000000.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None


In [58]:
lm.fit(data[cols], data.direction)

,estimator,LogisticRegre...x_iter=100000)
,n_jobs,None
,verbose,0
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1000000.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None


In [18]:
lm

,estimator,LogisticRegre...x_iter=100000)
,n_jobs,None
,verbose,0
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1000000.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None


In [59]:
data["pred"] = lm.predict(data[cols])

In [20]:
data

,price,returns,direction,lag1,lag2,lag3,lag4,lag5,pred
time,,,,,,,,,
2021-01-10 22:30:00,1.22244,-0.000204,-1.0,2.256259,0.056082,-2.227712,2.813520,0.390485,-1.0
2021-01-10 22:35:00,1.22199,-0.000368,-1.0,-0.695744,2.256259,0.056086,-2.227695,2.813518,-1.0
2021-01-10 22:40:00,1.22179,-0.000164,-1.0,-1.252986,-0.695744,2.256249,0.056079,-2.227695,1.0
2021-01-10 22:45:00,1.22140,-0.000319,-1.0,-0.556830,-1.252987,-0.695734,2.256218,0.056078,1.0
2021-01-10 22:50:00,1.22172,0.000262,1.0,-1.086425,-0.556830,-1.252974,-0.695734,2.256217,1.0
...,...,...,...,...,...,...,...,...,...
2025-12-16 23:35:00,1.17500,-0.000009,-1.0,0.261112,0.029335,-0.347297,0.203156,0.000358,-1.0
2025-12-16 23:40:00,1.17512,0.000102,1.0,-0.028610,0.261111,0.029340,-0.347300,0.203155,1.0
2025-12-16 23:45:00,1.17496,-0.000136,-1.0,0.348000,-0.028610,0.261115,0.029333,-0.347301,-1.0


In [60]:
hits = np.sign(data.direction * data.pred).value_counts()

In [61]:
hits

 1.0    801733
-1.0    700328
 0.0    587471
Name: count, dtype: int64

In [62]:
hit_ratio = hits[1.0] / sum(hits)
hit_ratio

np.float64(0.38369022345673576)

In [25]:
lm

,estimator,LogisticRegre...x_iter=100000)
,n_jobs,None
,verbose,0
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1000000.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None


In [27]:
import pickle

In [28]:
pickle.dump(lm, open(r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\20251217_logreg.pkl', "wb"))

In [29]:
params = {"mu":means, "std":stand_devs}
params

{'mu': lag1   -1.056889e-07
 lag2   -1.054011e-07
 dtype: float64,
 'std': lag1    0.000294
 lag2    0.000294
 dtype: float64}

In [30]:
pickle.dump(params, open(r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\20251217_params.pkl', "wb"))

In [32]:
class MLTrader(tpqoa.tpqoa):
    def __init__(self, conf_file, instrument, bar_length, lags, model, units):
        super().__init__(conf_file)
        self.instrument = instrument
        self.bar_length = pd.to_timedelta(bar_length)
        self.tick_data = pd.DataFrame()
        self.raw_data = None
        self.data = None
        self.last_bar = None
        self.units = units
        self.position = 0
        self.profits = []

        #*****************add strategy-specific attributes here******************
        self.lags = lags
        self.model = model
        #************************************************************************

    def get_most_recent(self, days = 5):
        while True:
            time.sleep(2)
            now = datetime.now(timezone.utc).replace(tzinfo=None)
            now = now - timedelta(microseconds = now.microsecond)
            past = now - timedelta(days = days)
            df = self.get_history(instrument = self.instrument, start = past, end = now,
                                   granularity = "S5", price = "M", localize = False).c.dropna().to_frame()
            df.rename(columns = {"c":self.instrument}, inplace = True)
            df = df.resample(self.bar_length, label = "right").last().dropna().iloc[:-1]
            self.raw_data = df.copy()
            self.last_bar = self.raw_data.index[-1]
            if pd.to_datetime(datetime.now(timezone.utc)) - self.last_bar < self.bar_length:
                break

    def on_success(self, time, bid, ask):
        print(self.ticks, end = " ")

        recent_tick = pd.to_datetime(time)
        df = pd.DataFrame({self.instrument:(ask + bid)/2},
                          index = [recent_tick])
        self.tick_data = pd.concat([self.tick_data, df]) # new with pd.concat()

        if recent_tick - self.last_bar > self.bar_length:
            self.resample_and_join()
            self.define_strategy()
            self.execute_trades()

    def resample_and_join(self):
        self.raw_data = pd.concat([self.raw_data, self.tick_data.resample(self.bar_length,
                                                                          label="right").last().ffill().iloc[:-1]])
        self.tick_data = self.tick_data.iloc[-1:]
        self.last_bar = self.raw_data.index[-1]

    def define_strategy(self): # "strategy-specific"
        df = self.raw_data.copy()

        #******************** define your strategy here ************************
        df = pd.concat([df, self.tick_data]) # new with pd.concat
        df["returns"] = np.log(df[self.instrument] / df[self.instrument].shift())
        cols = []
        for lag in range(1, self.lags + 1):
            col = f'lag{lag}'
            df[col] = df.returns.shift(lag)
            cols.append(col)
        df.dropna(inplace = True)

        df[cols] = (df[cols] - means) / stand_devs # newly added (scaling)

        df["position"] = lm.predict(df[cols])
        #***********************************************************************

        self.data = df.copy()

    def execute_trades(self):
        if self.data["position"].iloc[-1] == 1:
            if self.position == 0:
                order = self.create_order(self.instrument, self.units, suppress = True, ret = True)
                self.report_trade(order, "GOING LONG")
            elif self.position == -1:
                order = self.create_order(self.instrument, self.units * 2, suppress = True, ret = True)
                self.report_trade(order, "GOING LONG")
            self.position = 1
        elif self.data["position"].iloc[-1] == -1:
            if self.position == 0:
                order = self.create_order(self.instrument, -self.units, suppress = True, ret = True)
                self.report_trade(order, "GOING SHORT")
            elif self.position == 1:
                order = self.create_order(self.instrument, -self.units * 2, suppress = True, ret = True)
                self.report_trade(order, "GOING SHORT")
            self.position = -1
        elif self.data["position"].iloc[-1] == 0:
            if self.position == -1:
                order = self.create_order(self.instrument, self.units, suppress = True, ret = True)
                self.report_trade(order, "GOING NEUTRAL")
            elif self.position == 1:
                order = self.create_order(self.instrument, -self.units, suppress = True, ret = True)
                self.report_trade(order, "GOING NEUTRAL")
            self.position = 0

    def report_trade(self, order, going):
        time = order["time"]
        units = order["units"]
        price = order["price"]
        pl = float(order["pl"])
        self.profits.append(pl)
        cumpl = sum(self.profits)
        print("\n" + 100* "-")
        print(f'{time} | {going}')
        print(f'{time} | units = {units} | price = {price} | P&L = {pl} | Cum P&L = {cumpl}')
        print(100 * "-" + "\n")

In [33]:
lm = pickle.load(open(r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\20251217_logreg.pkl', "rb"))
lm

,estimator,LogisticRegre...x_iter=100000)
,n_jobs,None
,verbose,0
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1000000.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None


In [34]:
params = pickle.load(open(r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\20251217_params.pkl', "rb"))
params

{'mu': lag1   -1.056889e-07
 lag2   -1.054011e-07
 dtype: float64,
 'std': lag1    0.000294
 lag2    0.000294
 dtype: float64}

In [35]:
means = params["mu"]
stand_devs = params["std"]

In [36]:
means

lag1   -1.056889e-07
lag2   -1.054011e-07
dtype: float64

In [37]:
stand_devs

lag1    0.000294
lag2    0.000294
dtype: float64

In [38]:
trader = MLTrader(r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\oanda.cfg', "EUR_USD", "S5", lags = 2, model = lm, units = 10000)

In [39]:
trader.model

,estimator,LogisticRegre...x_iter=100000)
,n_jobs,None
,verbose,0
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1000000.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None


In [40]:
trader.get_most_recent()
trader.stream_data(trader.instrument, stop = 300)
if trader.position != 0: # if we have a final open position
    close_order = trader.create_order(trader.instrument, units = -trader.position * trader.units,
                                      suppress = True, ret = True)
    trader.report_trade(close_order, "GOING NEUTRAL")
    trader.position = 0

1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 
----------------------------------------------------------------------------------------------------
2025-12-17T12:15:02.915654690Z | GOING SHORT
2025-12-17T12:15:02.915654690Z | units = -10000.0 | price = 1.17154 | P&L = 0.0 | Cum P&L = 0.0
----------------------------------------------------------------------------------------------------

50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191

In [95]:
trader.data.tail(10)

,EUR_USD,returns,lag1,lag2,position
2025-12-17 10:25:00+00:00,1.171500,-0.000051,-0.319242,-1.074444,1.0
2025-12-17 10:30:00+00:00,1.171540,0.000034,-0.173981,-0.319243,1.0
2025-12-17 10:35:00+00:00,1.171740,0.000171,0.116588,-0.173982,1.0
2025-12-17 10:40:00+00:00,1.171710,-0.000026,0.581441,0.116587,-1.0
2025-12-17 10:45:00+00:00,1.171740,0.000026,-0.086796,0.581440,-1.0
2025-12-17 10:50:00+00:00,1.172080,0.000290,0.087516,-0.086797,1.0
2025-12-17 10:55:00+00:00,1.172400,0.000273,0.987970,0.087515,-1.0
2025-12-17 11:00:00+00:00,1.172510,0.000094,0.929614,0.987969,-1.0
2025-12-17 11:05:00+00:00,1.172385,-0.000107,0.319732,0.929613,-1.0
2025-12-17 11:05:02.030385711+00:00,1.172380,-0.000004,-0.362566,0.319731,1.0


In [96]:
trader.tick_data

,EUR_USD
2025-12-17 11:05:02.030385711+00:00,1.172380
2025-12-17 11:05:02.474713396+00:00,1.172380
2025-12-17 11:05:03.010623494+00:00,1.172380
2025-12-17 11:05:05.017600781+00:00,1.172380
2025-12-17 11:05:06.095221719+00:00,1.172390
...,...
2025-12-17 11:07:06.081560777+00:00,1.172340
2025-12-17 11:07:09.170358023+00:00,1.172325
2025-12-17 11:07:10.107773340+00:00,1.172325
2025-12-17 11:07:10.222884767+00:00,1.172315
